In [1]:
import marimo as mo
import pandas as pd
import numpy as np

# Preparación del dataset

In [2]:

DROP_COLUMNS = [
    "stitch_display",
    "duet_display",
    "city",
    "poi_tt_type_name_super",
    "poi_tt_type_code",
    "country_code",
]

BOOLEAN_FEATURES = [
    "duet_enabled",
    "is_ad",
    "item_mute",
    "item_control_can_repost",
    "official_item",
    "original_item",
    "share_enabled",
    "stitch_enabled",
    "music_original",
]
NOMINAL_FEATURES = [
    "desc",
    "address",
    "poi_name",
    "city_code",
    "poi_category",
    "poi_tt_type_name_medium",
    "poi_tt_type_name_tiny",
    "challenges",
    "music_id",
    "music_title",
    "duet_info_duet_from_id",
    "music_author_name",
    "poi_id",
    "diversification_id",
    "item_comment_status",
]
NUMERICAL_FEATURES = [
    "music_duration",
    "vq_score",
    "duration",
]
TARGET = "play_count"

ID_FEATURES = [c for c in NOMINAL_FEATURES if c.endswith("id")]
KEEP_COLUMNS = BOOLEAN_FEATURES + NOMINAL_FEATURES + NUMERICAL_FEATURES + [TARGET]

In [3]:
df_raw = pd.read_parquet("../data/raw/sample_data.parquet")

## 1. Selección de variables

In [4]:
# Dropeo explícito y luego filtro a las columnas que sí queremos
df_1 = df_raw.drop(columns=[c for c in DROP_COLUMNS if c in df_raw.columns])
faltantes = [c for c in KEEP_COLUMNS if c not in df_1.columns]
if faltantes:
    raise KeyError(f"Columnas esperadas que no están en el dataset: {faltantes}")
df_1 = df_1[KEEP_COLUMNS].copy()
df_1.shape

(2000000, 28)

## 2. Tipos de dato

In [5]:
# no no vamo a engañar, esta preparacion ta batante vaicodea 
df_2 = df_1.copy()

# Booleanas: primero a dtype nullable "boolean" para no convertir NaN -> True
# (astype(bool) directo hace eso). El cast final a bool va al terminar la limpieza.
_map = {
    "true": True, "false": False, "1": True, "0": False,
    "yes": True, "no": False, "t": True, "f": False,
}
for _c in BOOLEAN_FEATURES:
    _s = df_2[_c]
    if _s.dtype == object:
        _s = _s.astype(str).str.strip().str.lower().map(_map)
    df_2[_c] = _s.astype("bool")

# *id -> category (más ligero que object y semánticamente correcto)
for _c in ID_FEATURES:
    df_2[_c] = df_2[_c].astype("string").astype("category")

# Resto de nominales -> object (string nullable)
for _c in [c for c in NOMINAL_FEATURES if c not in ID_FEATURES]:
    df_2[_c] = df_2[_c].astype("string")

# Numéricas y target -> numérico
for _c in NUMERICAL_FEATURES + [TARGET]:
    df_2[_c] = pd.to_numeric(df_2[_c], errors="coerce")

df_2.dtypes

duet_enabled                   bool
is_ad                          bool
item_mute                      bool
item_control_can_repost        bool
official_item                  bool
original_item                  bool
share_enabled                  bool
stitch_enabled                 bool
music_original                 bool
desc                         string
address                      string
poi_name                     string
city_code                    string
poi_category                 string
poi_tt_type_name_medium      string
poi_tt_type_name_tiny        string
challenges                   string
music_id                   category
music_title                  string
duet_info_duet_from_id     category
music_author_name            string
poi_id                     category
diversification_id         category
item_comment_status          string
music_duration              float64
vq_score                    float64
duration                      int64
play_count                  

## 3. Imputación de numéricas

In [6]:
# Diagnóstico previo: % nulos y asimetría de cada numérica
resumen_num = df_2[NUMERICAL_FEATURES].agg(["count", "mean", "median", "skew"]).T
resumen_num["pct_nulos"] = df_2[NUMERICAL_FEATURES].isna().mean() * 100
resumen_num

,count,mean,median,skew,pct_nulos
music_duration,1998007.0,73.885832,60.00,6.638729,0.09965
vq_score,2000000.0,61.358796,66.19,-2.837452,0.00000
duration,2000000.0,53.830574,29.00,8.260611,0.00000


In [7]:
# Estrategia (Little & Rubin; Hastie et al.): para univariada simple, la media solo
# es razonable con distribución ~simétrica; con asimetría fuerte (|skew| > 1) la
# mediana es el estimador robusto. Se decide por columna.
df_3 = df_2.copy()
imputacion = {}
for _c in NUMERICAL_FEATURES:
    _skew = df_3[_c].skew(skipna=True)
    if np.isnan(_skew) or abs(_skew) > 1:
        _valor, _estrategia = df_3[_c].median(), "median"
    else:
        _valor, _estrategia = df_3[_c].mean(), "mean"
    _n = int(df_3[_c].isna().sum())
    df_3[_c] = df_3[_c].fillna(_valor)
    imputacion[_c] = {"estrategia": _estrategia, "valor": _valor, "imputados": _n}
imputacion

{'music_duration': {'estrategia': 'median',
  'valor': np.float64(60.0),
  'imputados': 1993},
 'vq_score': {'estrategia': 'median',
  'valor': np.float64(66.19),
  'imputados': 0},
 'duration': {'estrategia': 'median',
  'valor': np.float64(29.0),
  'imputados': 0}}

## 4. Nulos residuales

In [8]:
nulos_restantes = df_3.isna().sum()
nulos_restantes[nulos_restantes > 0]

desc                       23702
address                        1
city_code                      3
poi_tt_type_name_tiny      55065
music_id                    1904
music_title                  523
duet_info_duet_from_id      1003
music_author_name           4321
diversification_id        221874
dtype: int64

In [9]:
n_antes = len(df_3)
df_final = df_3.dropna().copy()

# Ahora sí, cast final a bool nativo (ya sin nulos)
for _c in BOOLEAN_FEATURES:
    df_final[_c] = df_final[_c].astype(bool)

# Limpia categorías huérfanas tras el dropna
for _c in df_final.select_dtypes("category").columns:
    df_final[_c] = df_final[_c].cat.remove_unused_categories()

print(f"Filas eliminadas: {n_antes - len(df_final)} ({(n_antes - len(df_final)) / n_antes:.2%})")
df_final.shape

Filas eliminadas: 295811 (14.79%)


(1704189, 28)

In [10]:
df_final.dtypes

duet_enabled                   bool
is_ad                          bool
item_mute                      bool
item_control_can_repost        bool
official_item                  bool
original_item                  bool
share_enabled                  bool
stitch_enabled                 bool
music_original                 bool
desc                         string
address                      string
poi_name                     string
city_code                    string
poi_category                 string
poi_tt_type_name_medium      string
poi_tt_type_name_tiny        string
challenges                   string
music_id                   category
music_title                  string
duet_info_duet_from_id     category
music_author_name            string
poi_id                     category
diversification_id         category
item_comment_status          string
music_duration              float64
vq_score                    float64
duration                      int64
play_count                  

In [11]:
df_final.head()

,duet_enabled,is_ad,item_mute,item_control_can_repost,official_item,original_item,share_enabled,stitch_enabled,music_original,desc,...,music_title,duet_info_duet_from_id,music_author_name,poi_id,diversification_id,item_comment_status,music_duration,vq_score,duration,play_count
5261844,True,True,True,True,True,True,True,True,True,د خوست ولایت ځنګلونه وهل کیږي. دا له هیواد او ...,...,original sound,0.0,Sawabdin Makhkash,22535865200907089,10083.0,0,47.0,47.11,47,2455
9576919,True,True,True,True,True,True,True,True,True,May Book Reviews. My Vela Scarves stay on with...,...,Manifestation,0.0,"Perfect, so dystopian",21568226288236304,10014.0,0,209.0,0.00,32,359
1206048,True,True,True,True,True,True,True,True,True,#فالكونز #عامر,...,original sound,0.0,يزن / فالكونز,22535865200907089,10075.0,0,61.0,57.42,61,113638
6206188,True,True,True,True,True,True,True,True,True,#therealeannise #sassysexysilverfox #over50 #d...,...,Diamonds Are Forever (From Diamonds Are Forever),0.0,The Soundtrack Tribute Band,22535865206966264,10029.0,0,60.0,62.21,46,479
8146649,True,True,True,True,True,True,True,True,True,"Breastfeeding as a birth control method? Tips,...",...,original sound,0.0,Kelley✨| SAHM of 3 ☀️🌸🌼| RN,22535865200907089,10018.0,0,156.0,70.97,156,1353


In [12]:
# Descomenta para guardar el resultado
# df_final.to_parquet("data_clean.parquet", index=False)
df_final.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
duet_enabled,1704189,1,True,1704189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_ad,1704189,1,True,1704189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
item_mute,1704189,1,True,1704189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
item_control_can_repost,1704189,1,True,1704189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
official_item,1704189,1,True,1704189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_item,1704189,1,True,1704189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
share_enabled,1704189,1,True,1704189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
stitch_enabled,1704189,1,True,1704189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
music_original,1704189,1,True,1704189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
desc,1704189,1680901,#fyp,506,NaN,NaN,NaN,NaN,NaN,NaN,NaN
